# Bayesian POLR baseline — reproduction of Semenova et al. (2020)

This notebook reproduces the **proportional odds logistic regression (POLR) baseline**
from:

> Semenova, Williams, Afzal & Lazic (2020), *A Bayesian neural network for toxicity
> prediction*, Computational Toxicology 16:100133 (`original_research.pdf`).

The paper's **baseline model is the Bayesian POLR**; the proposed model is a BNN.
The paper reports (Table 2):

| Metric | POLR baseline |
|---|---|
| WAIC | 267.3 |
| Mean OBS train / test | 0.14 / 0.16 |
| Median OBS train / test | 0.11 / 0.12 |
| Mean BSS train / test | 0.24 / 0.20 |
| Median BSS train / test | 0.36 / 0.36 |
| BA train / test | 0.61 / 0.61 |

## Reproduction notes

The paper's code is Julia/Turing and was not released, so an exact bit-for-bit match
is not possible. This notebook matches the *methodology*:

1. **Design matrix** — 8 main effects + 21 pairwise interactions (interactions with
   Cmax excluded), standardised. The paper's Eq. 1 describes a raw `147 x 29` design
   matrix but never mentions scaling; raw inputs are numerically impossible here
   (`max|x| = 90000` gives `-inf` log-probability at initialisation), so the paper's
   inputs must have been scaled.
2. **Model** — `w ~ Normal(0, sigma^2)`, `sigma ~ HalfNormal(1)`,
   `cutpoints ~ Normal(0, 20)` ordered, `y ~ OrderedLogistic(eta, cutpoints)`.
3. **Metrics as posterior distributions** — OBS, BSS and BA are computed for every
   posterior draw and reported as mean and median, matching the paper's Table 2 and
   Figure 9/10.
4. **WAIC** — standard `-2 (lppd - p_waic)` from the posterior. ArviZ 1.x removed
   `az.waic`, so it is computed manually.
5. **Bootstrap** — 20 resamples of the 147 training compounds, evaluated out-of-sample
   and on the test set (paper Table 3, Appendix E).

### Numerical fixes required by PyMC 6.3
- `pm.OrderedLogistic` underflows to `-inf` for large `|eta|` allowed by the wide
  `Normal(0, 20)` cutpoint prior, which aborts NUTS. A mathematically identical
  log1mexp formulation is used via `pm.Potential`.
- The distribution-level `initval` on `cutpoints` prevents PyMC from computing the
  log-likelihood (`Cannot convert models with non-default initial_values`), so the
  starting point is passed to `pm.sample` instead.


In [ ]:
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import arviz as az
from scipy.special import expit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PolynomialFeatures, StandardScaler


## 1. Load data

147 training and 37 test compounds, severity classes 1 (safe), 2 (moderate), 3 (most DILI concern).

In [ ]:
train_df = pd.read_parquet("../data/01_raw/train_df.parquet")
test_df = pd.read_parquet("../data/01_raw/test_df.parquet")

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)
print("\nTraining severity distribution:")
print(train_df["dili_sev"].value_counts().sort_index())
print("\nTesting severity distribution:")
print(test_df["dili_sev"].value_counts().sort_index())


## 2. Design matrix

All 8 main effects plus the 21 pairwise interactions among the seven non-Cmax
predictors (`log10cmax` enters as a main effect only), then standardised.


In [ ]:
INTERACTION_FEATURES = ["BSEP", "ClogP", "Fsp3", "Glu", "Glu_Gal", "HepG2", "THLE"]
PASS_THROUGH = ["log10cmax"]
DROP = ["Drug", "Unnamed: 0", "vDILIConcern", "dili_sev"]


def make_design(train, test):
    features = [c for c in train.columns if c not in DROP]
    transformer = ColumnTransformer(
        transformers=[
            ("poly_interaction",
             PolynomialFeatures(degree=2, interaction_only=True, include_bias=False),
             INTERACTION_FEATURES),
            ("pass_through", "passthrough", PASS_THROUGH),
        ]
    )
    transformer.fit(train[features])
    scaler = StandardScaler().fit(transformer.transform(train[features]))
    X_train = scaler.transform(transformer.transform(train[features]))
    X_test = scaler.transform(transformer.transform(test[features]))
    return X_train, X_test


X_train, X_test = make_design(train_df, test_df)
y_train = train_df["dili_sev"].to_numpy(int)
y_test = test_df["dili_sev"].to_numpy(int)

print("Design matrix train / test:", X_train.shape, X_test.shape)


## 3. Model

Numerically stable ordered-logistic likelihood (identical maths to
`pm.OrderedLogistic`), then the Bayesian POLR.


In [ ]:
def log_sigmoid(x):
    return -pt.softplus(-x)


def ordered_logistic_logp(eta, c0, c1, y0):
    """Stable log P(y | eta, cutpoints) for categories 0, 1, 2."""
    b1 = c0 - eta
    b2 = c1 - eta
    ls1 = log_sigmoid(b1)
    ls2 = log_sigmoid(b2)
    lp2 = ls2 + pt.log(-pt.expm1(ls1 - ls2))
    return pt.where(pt.eq(y0, 0), ls1,
                    pt.where(pt.eq(y0, 1), lp2, log_sigmoid(-b2)))


def fit_polr(X, y0, draws=2000, tune=1000, chains=4,
             target_accept=0.95, seed=42):
    with pm.Model() as model:
        sigma = pm.HalfNormal("sigma", sigma=1.0)
        w = pm.Normal("w", mu=0, sigma=sigma, shape=X.shape[1])
        eta = pt.dot(X, w)
        cutpoints = pm.Normal(
            "cutpoints", mu=0, sigma=20, shape=2,
            transform=pm.distributions.transforms.ordered,
            initval=np.array([-0.5, 0.5]),
        )
        pm.Potential(
            "y_obs",
            ordered_logistic_logp(eta, cutpoints[0], cutpoints[1], y0).sum(),
        )
        trace = pm.sample(
            draws=draws, tune=tune, chains=chains,
            target_accept=target_accept, random_seed=seed,
            progressbar=True, init="adapt_diag",
            initvals={"w": np.zeros(X.shape[1]),
                      "cutpoints": np.array([-0.5, 0.5])},
        )
    return model, trace


## 4. Fit the full model on the training data

In [ ]:
model, trace = fit_polr(X_train, y_train - 1)


In [ ]:
az.summary(trace, var_names=["sigma", "cutpoints"], round_to=3)


## 5. Prediction and metrics

OBS, BSS, BA are computed for **every posterior draw**; `y` is 1-indexed.

In [ ]:
def predict_polr(trace, X):
    w = trace.posterior["w"].values.reshape(-1, X.shape[1])
    c = trace.posterior["cutpoints"].values.reshape(-1, 2)
    eta = X @ w.T
    p1 = expit(c[:, 0][None, :] - eta)
    p2 = expit(c[:, 1][None, :] - eta) - p1
    p3 = 1.0 - expit(c[:, 1][None, :] - eta)
    return {"p1": p1, "p2": p2, "p3": p3}


def ordered_brier_score(y, p1, p2, p3):
    o1 = (y == 1).astype(float)[:, None]
    o2 = (y <= 2).astype(float)[:, None]
    return (((p1 - o1) ** 2 + (p1 + p2 - o2) ** 2) / 2).mean(axis=0)


def reference_obs(y):
    f = np.bincount(y, minlength=4)[1:4] / len(y)
    return ordered_brier_score(
        y, np.full((len(y), 1), f[0]),
        np.full((len(y), 1), f[1]), np.full((len(y), 1), f[2]))[0]


def brier_skill_score(y, p1, p2, p3):
    return 1 - ordered_brier_score(y, p1, p2, p3) / reference_obs(y)


def balanced_accuracy(y, p1, p2, p3):
    pred = np.argmax(np.stack([p1, p2, p3], 0), 0) + 1
    return np.mean([(pred[y == k] == k).sum(0) / (y == k).sum()
                    for k in (1, 2, 3)], axis=0)


def waic(trace, X, y):
    w = trace.posterior["w"].values.reshape(-1, X.shape[1])
    c = trace.posterior["cutpoints"].values.reshape(-1, 2)
    eta = X @ w.T
    s1 = expit(c[:, 0][None, :] - eta)
    s2 = expit(c[:, 1][None, :] - eta)
    p = np.stack([s1, s2 - s1, 1.0 - s2], axis=-1)
    n, s = eta.shape
    logp = np.log(np.clip(
        p[np.arange(n)[:, None], np.arange(s)[None, :], (y - 1)[:, None]],
        1e-300, None))
    m = logp.max(1, keepdims=True)
    lppd = (m[:, 0] + np.log(np.exp(logp - m).mean(1))).sum()
    return -2 * (lppd - logp.var(1, ddof=1).sum())


In [ ]:
rows = []
for name, X, y in [("train", X_train, y_train), ("test", X_test, y_test)]:
    pred = predict_polr(trace, X)
    obs = ordered_brier_score(y, pred["p1"], pred["p2"], pred["p3"])
    bss = brier_skill_score(y, pred["p1"], pred["p2"], pred["p3"])
    ba = balanced_accuracy(y, pred["p1"], pred["p2"], pred["p3"])
    rows.append({"set": name,
                 "OBS_mean": obs.mean(), "OBS_median": np.median(obs),
                 "BSS_mean": bss.mean(), "BSS_median": np.median(bss),
                 "BA_mean": ba.mean(), "BA_median": np.median(ba)})

full_metrics = pd.DataFrame(rows).set_index("set")
print(full_metrics.round(3).to_string())
print(f"\nWAIC (full model): {waic(trace, X_train, y_train):.1f}")

print("\nPaper POLR baseline (Table 2):")
print("  WAIC 267.3")
print("  OBS  mean train/test 0.14 / 0.16 | median 0.11 / 0.12")
print("  BSS  mean train/test 0.24 / 0.20 | median 0.36 / 0.36")
print("  BA   train/test 0.61 / 0.61")


## 6. Calibration (paper Appendix D)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

test_pred = predict_polr(trace, X_test)
p_2plus3 = (test_pred["p2"] + test_pred["p3"]).mean(axis=1)
p_3 = test_pred["p3"].mean(axis=1)

prob_true_23, prob_pred_23 = calibration_curve(
    (y_test >= 2).astype(int), p_2plus3, n_bins=5, strategy="quantile")
prob_true_3, prob_pred_3 = calibration_curve(
    (y_test == 3).astype(int), p_3, n_bins=5, strategy="quantile")

plt.figure(figsize=(7, 6))
plt.plot(prob_pred_23, prob_true_23, marker="o", label="Category 1 vs 2+3")
plt.plot(prob_pred_3, prob_true_3, marker="o", label="Categories 1+2 vs 3")
plt.plot([0, 1], [0, 1], "--", label="Perfect calibration")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed proportion")
plt.title("POLR calibration (test set)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 7. Bootstrap experiments (paper Table 3, Appendix E)

Each iteration resamples the 147 training compounds with replacement, fits a fresh
POLR on the in-sample set, and evaluates on the out-of-sample (unselected) and test
sets. Reported as median (sd), matching the paper.

> This loop fits 20 full Bayesian models and takes a long time; it is not executed
> in the saved notebook.


In [ ]:
n_bootstrap = 20
rng = np.random.default_rng(42)
bootstrap_results = []


In [ ]:
for b in range(n_bootstrap):
    idx = rng.choice(len(train_df), size=len(train_df), replace=True)
    oos_idx = np.setdiff1d(np.arange(len(train_df)), np.unique(idx))

    # ensure all three classes are present in-sample
    while len(np.unique(train_df["dili_sev"].to_numpy()[idx])) < 3:
        idx = rng.choice(len(train_df), size=len(train_df), replace=True)
        oos_idx = np.setdiff1d(np.arange(len(train_df)), np.unique(idx))

    boot_df = train_df.iloc[idx]
    oos_df = train_df.iloc[oos_idx]

    # refit preprocessing on the in-sample data only
    features = [c for c in train_df.columns if c not in DROP]
    transformer = ColumnTransformer(
        transformers=[
            ("poly_interaction",
             PolynomialFeatures(degree=2, interaction_only=True, include_bias=False),
             INTERACTION_FEATURES),
            ("pass_through", "passthrough", PASS_THROUGH),
        ])
    transformer.fit(boot_df[features])
    scaler = StandardScaler().fit(transformer.transform(boot_df[features]))

    X_boot = scaler.transform(transformer.transform(boot_df[features]))
    X_oos = scaler.transform(transformer.transform(oos_df[features]))
    X_test_b = scaler.transform(transformer.transform(test_df[features]))

    y_boot = boot_df["dili_sev"].to_numpy(int)
    y_oos = oos_df["dili_sev"].to_numpy(int)

    _, trace_b = fit_polr(X_boot, y_boot - 1, seed=42 + b)

    result = {"bootstrap": b + 1, "N_unique_train": len(np.unique(idx)),
              "N_oos": len(oos_idx), "WAIC": waic(trace_b, X_boot, y_boot)}
    for tag, X, y in [("oos", X_oos, y_oos), ("test", X_test_b, y_test)]:
        pred = predict_polr(trace_b, X)
        obs = ordered_brier_score(y, pred["p1"], pred["p2"], pred["p3"])
        result[f"OBS_{tag}"] = np.median(obs)
        result[f"BSS_{tag}"] = np.median(
            brier_skill_score(y, pred["p1"], pred["p2"], pred["p3"]))
        result[f"BA_{tag}"] = np.median(
            balanced_accuracy(y, pred["p1"], pred["p2"], pred["p3"]))

    centroid = X_boot.mean(axis=0)
    result["test_centroid_dist"] = np.mean(
        np.linalg.norm(X_test_b - centroid, axis=1))
    result["oos_centroid_dist"] = np.mean(
        np.linalg.norm(X_oos - centroid, axis=1))

    bootstrap_results.append(result)
    print(f"bootstrap {b + 1}/{n_bootstrap} done")


In [ ]:
bootstrap_df = pd.DataFrame(bootstrap_results)
summary = []
for metric in ["WAIC", "OBS_oos", "OBS_test", "BSS_oos", "BSS_test",
               "BA_oos", "BA_test"]:
    summary.append({"Metric": metric,
                    "Median": bootstrap_df[metric].median(),
                    "SD": bootstrap_df[metric].std()})
bootstrap_summary = pd.DataFrame(summary).set_index("Metric")
print(bootstrap_summary.round(3).to_string())

print("\nPaper POLR baseline (Table 3, median (sd)):")
print("  OBS out-of-sample/test 0.12 (0.02) / 0.12 (0.02)")
print("  BSS out-of-sample/test 0.34 (0.11) / 0.35 (0.08)")
print("  BA  out-of-sample/test 0.53 (0.05) / 0.56 (0.04)")


## 8. Save results

In [ ]:
full_metrics.to_csv("../data/08_reporting/polr_full_metrics.csv")
bootstrap_df.to_csv("../data/08_reporting/polr_bootstrap_results.csv", index=False)
bootstrap_summary.to_csv("../data/08_reporting/polr_bootstrap_summary.csv")
print("Saved to data/08_reporting/")


## 9. Centroid distance vs performance (paper Figure 11)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, metric in zip(axes, ["OBS_test", "BSS_test", "BA_test"]):
    ax.scatter(bootstrap_df["test_centroid_dist"], bootstrap_df[metric], s=40)
    r = np.corrcoef(bootstrap_df["test_centroid_dist"], bootstrap_df[metric])[0, 1]
    ax.set_xlabel("Mean distance of test set to in-sample centroid")
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.grid(alpha=0.3)
    ax.annotate(f"r = {r:.3f}", xy=(0.98, 0.05), xycoords="axes fraction",
                ha="right", fontsize=11)
plt.tight_layout()
plt.show()
